In [ ]:
import networkx as nx
import pandas as pd
import urllib

In [ ]:
with urllib.request.urlopen("https://raw.githubusercontent.com/Somon8/social_graphs_25/main/final-project/graphs/df_votes_enriched.csv") as response:
    df = pd.read_csv(response)

In [ ]:
# For each unique politician create a DF
# For each other unique politician
# Count how many voting id´s they share
# Count how many voting id's they have the same value in
# Cakculate agreement percentage
# Add it as edges to a graph
G = nx.Graph()
#Let's do some agreement stuff
# def create_network_from_theme(df):
#     return G

#WE could also already have filtered the dataframe here by theme or something like that. It's probably the smartest thing to do.
politician_list = df['politician'].unique()
for idx, politician in enumerate(politician_list):

    df_pol = df[df['politician'] == politician][['voting_id', 'vote_type']]

    for other_politician in politician_list[idx+1:]: #Look at remaining politicians to avoid double work
        df_other = df[df['politician'] == other_politician][['voting_id', 'vote_type']]
        
        #WE NEED TO ADD SOMETHING ABOUT THE THEME HERE?
        total_df = df_pol.merge(df_other, on='voting_id', how='inner') #Count how many voting_id's they share
        agree_df = df_pol.merge(df_other, on=['voting_id', 'vote_type'], how='inner') #Count share where they also voted the same
        total = len(total_df)
        agree = len(agree_df)

        agree_percent = agree / total if total > 0 else 0
        G.add_edge(politician, other_politician, weight=agree_percent)

In [ ]:
# df.to_csv("graphs/node_df.csv", index = True)

In [ ]:
# DOES THIS EVEN MAKE SENSE?
g_df = nx.to_pandas_edgelist(G)

# g_df['weight']
g_df[g_df['target'] == "Aki-Matilda Høegh-Dam"]
# df_draw[df_draw['target'] == "Aki-Matilda Høegh-Dam"]
# g_df[g_df['weight'] > agreement_threshold]

In [ ]:
figure, axs = plt.subplots(1,1,figsize = (18,10)) #Width, Height

pos = nx.forceatlas2_layout(G
                                        , seed = 42
                                        , weight = "weight"
                                        )

# nodes
agreement_threshold = 0.9
df_draw = g_df[g_df['weight'] > agreement_threshold]

print(len(g_df), len(df_draw))

#Get the edges and widths to draw
edges = list(zip(df_draw['source'], df_draw['target']))
edge_widths = df_draw['weight'].tolist()


nx.draw_networkx_nodes(G, pos, node_size=200, ax = axs)

# edges
nx.draw_networkx_edges(G, pos, edgelist=edges, width=[w * 1 for w in edge_widths], alpha = 0.3, ax = axs)


# node labels
nx.draw_networkx_labels(G, pos, font_size=10, ax = axs)


# ax.margins(0.08)
plt.axis("off")
plt.tight_layout()
plt.show()